# Paper 3, pass 2: does the checkpoint predict these recordings?

Pass 1 measured the ceiling from the recordings alone. Its first result was
withdrawn: it read `list(f.keys())[0]`, which is episode `s01e02a` rather than
the episode this pass predicts, and its transpose guard could not fire on a
`(482, 1000)` array, so every correlation ran across the parcel axis. The
corrected ceiling for `s01e01a` is `0.1517`, CI `[0.1484, 0.1551]`, four
subjects, 1000 of 1000 parcels, 592 timepoints, from
`scripts/paper3_noise_ceiling.py`. The withdrawn figure was `0.2247`.

This pass runs the released checkpoint over the same stimulus and correlates
its predictions against those recordings, matching the episode by name and
asserting orientation against the atlas size.

Committed before the run: the sign is the finding, nothing is flipped or
absolute-valued, and a parcel-level `r` is not comparable with the audit's
vertex-level `-0.0145` because averaging within a parcel raises correlations.

**Attach `ckadirt/algonauts2025nsl`. GPU required.**

In [ ]:
%%bash
# Same pinned build as the corpus scan: tribev2 declares torch>=2.5.1,<2.7 and Kaggle
# ships newer, so the model would otherwise run against a version it was never built for.
set -e
nvidia-smi --query-gpu=name --format=csv,noheader | head -1 | sed 's/^/card: /'
pip install -q --index-url https://download.pytorch.org/whl/cu121 \
  torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1
python -c "import torch; print('torch', torch.__version__)"

In [ ]:
import glob
import os

roots = sorted(glob.glob('/kaggle/input/**/algonauts_2025.competitors', recursive=True))
root = roots[0]
print('root:', root)

# The stimulus this pass predicts. Find it before spending GPU time on setup: if the
# media is not present in the mirror, the run should stop here rather than at hour two.
media = sorted(
    glob.glob(os.path.join(root, 'stimuli', '**', '*.mkv'), recursive=True)
    + glob.glob(os.path.join(root, 'stimuli', '**', '*.mp4'), recursive=True)
)
print(f'stimulus files found: {len(media)}')
for path in media[:5]:
    print(f'  {os.path.getsize(path)/1e6:8.1f} MB  {os.path.relpath(path, root)}')

assert media, 'No stimulus media in this mirror. Pass 2 cannot run; report that and stop.'

In [ ]:
%%bash
set -e
mkdir -p /kaggle/temp
cd /kaggle/temp
rm -rf monarch tribev2
git clone -q --branch thesis/amendment-and-analysis-layer \
  https://github.com/brn-mwai/monarch.git monarch
git clone -q https://github.com/brn-mwai/tribev2.git
apt-get -qq update > /dev/null 2>&1 && apt-get -qq install -y ffmpeg > /dev/null
printf 'torch==2.5.1\ntorchvision==0.20.1\ntorchaudio==2.5.1\n' > /kaggle/temp/constraints.txt
pip install -q exca==0.5.21
pip install -q -c /kaggle/temp/constraints.txt /kaggle/temp/tribev2
pip install -q -c /kaggle/temp/constraints.txt "whisperx==3.4.2"
pip install -q -c /kaggle/temp/constraints.txt nltk nibabel ujson mne torchmetrics
pip install -q --no-deps "ctranslate2==4.5.0"
python -c "import neuralset, neuraltrain, tribev2; print('tribev2 stack imports OK')"

In [ ]:
import sys

sys.path.insert(0, '/kaggle/temp/monarch/services/inference')
from scripts.kaggle_bootstrap import apply_session_environment

SESSION = apply_session_environment()

In [ ]:
from pathlib import Path

import numpy as np

from app.services.inference import TribeInferenceService

STIMULUS = Path(media[0])
print('predicting:', STIMULUS.name)

service = TribeInferenceService()
service.load_model()
result = service.predict_video(STIMULUS)
prediction = np.asarray(result['raw_preds'], dtype=np.float32)
print('prediction shape:', prediction.shape)

np.save('/kaggle/working/prediction.npy', prediction)


In [ ]:
import h5py

from app.services.encoder_validation import bootstrap_ci, noise_ceiling, vertex_correlation
from app.services.parcellation import project_to_parcels

N_PARCELS = 1000
EPISODE = STIMULUS.stem.split('_')[-1]
print('episode:', EPISODE)

# The checkpoint emits fsaverage5 vertices; the recordings are Schaefer parcels. The
# prediction is projected into parcel space rather than the recordings being upsampled,
# since upsampling would invent within-parcel structure the data does not have.
labels = np.load('/kaggle/temp/monarch/services/inference/data/schaefer1000_fsaverage5.npy')
projected = project_to_parcels(prediction.T, labels, n_parcels=N_PARCELS)['parcel_timeseries']
print('projected:', projected.shape)


def oriented(data, path):
    """Return (parcels, timepoints), decided by the atlas size rather than by axis length.

    A `shape[0] > shape[1]` rule cannot fire on an episode with fewer TRs than parcels, and
    the silently transposed array still correlates and still looks plausible. Pass 1 shipped
    a number that way.
    """
    if data.shape[1] == N_PARCELS:
        return data.T
    if data.shape[0] == N_PARCELS:
        return data
    raise ValueError(f'{path}: neither axis is {N_PARCELS} parcels, shape {data.shape}')


# The episode is matched by name, never by position: the first key in these files is
# s01e02a, and subjects do not agree on which session an episode sits under.
responses = []
for path in sorted(p for p in glob.glob(os.path.join(root, 'fmri', '**', '*.h5'),
                                        recursive=True) if 'friends' in p):
    with h5py.File(path, 'r') as f:
        keys = [k for k in f.keys() if k.endswith(f'task-{EPISODE}')]
        assert len(keys) == 1, f'{os.path.basename(path)}: {len(keys)} keys for {EPISODE}'
        node = f[keys[0]]
        data = node[list(node.keys())[0]][:] if isinstance(node, h5py.Group) else node[:]
    responses.append(oriented(np.asarray(data, dtype=float), path))
    print(f'  {os.path.basename(path)[:24]}  {keys[0]}  {responses[-1].shape}')

assert responses, 'No friends recordings matched; pass 2 cannot run.'

shortest = min([projected.shape[1]] + [r.shape[1] for r in responses])
projected = projected[:, :shortest]
responses = [r[:, :shortest] for r in responses]
print('subjects:', len(responses), 'parcels:', responses[0].shape[0],
      'timepoints:', shortest)

In [ ]:
import json

per_subject = [vertex_correlation(projected, observed) for observed in responses]
means = np.array([r['mean_r'] for r in per_subject])
encoder = bootstrap_ci(means, n_resamples=10000, seed=0)
ceiling = noise_ceiling(responses)
ceiling_ci = bootstrap_ci(np.nanmean(ceiling['per_subject_r'], axis=1),
                          n_resamples=10000, seed=0)

print('Checkpoint against held-out subjects, parcel level')
print(f"  encoder mean r : {encoder['point']:+.4f}  "
      f"95% CI [{encoder['low']:+.4f}, {encoder['high']:+.4f}]")
print(f"  noise ceiling  : {ceiling_ci['point']:+.4f}  "
      f"95% CI [{ceiling_ci['low']:+.4f}, {ceiling_ci['high']:+.4f}]")
print(f"  beats zero     : {encoder['low'] > 0}")
print(f"  reaches ceiling: {encoder['low'] >= ceiling_ci['point']}")
print()
for i, value in enumerate(means):
    print(f'  subject {i + 1}: {value:+.4f}')

with open('/kaggle/working/paper3_validation.json', 'w') as handle:
    json.dump({
        'stimulus': STIMULUS.name,
        'encoder': encoder,
        'ceiling': ceiling_ci,
        'per_subject_mean_r': means.tolist(),
        'space': 'Schaefer 1000 parcels',
        'note': 'parcel level; not comparable with vertex-level r from the audit',
    }, handle, indent=2)
print('\nwrote /kaggle/working/paper3_validation.json')

## Reading this

Whatever the sign, it is reported as measured. A negative encoder correlation
against a positive ceiling would replicate the audit's direction in parcel
space; a positive one below the ceiling means the checkpoint carries signal but
less than people share with each other.

Neither number is comparable with `-0.0145` from the audit, which is
vertex-level. That comparison needs surface derivatives this mirror does not
carry.